In [ ]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "YOUR_API_KEY"
os.environ["LANGCHAIN_PROJECT"] = "resume-screening"

In [2]:
!pip install langchain transformers


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from transformers import pipeline

generator = pipeline("text-generation", model="sshleifer/tiny-gpt2")

def local_llm(prompt):
    result = generator(
        prompt,
        max_new_tokens=100,   
        do_sample=True
    )
    return result[0]['generated_text']

    # 🔥 REMOVE PROMPT FROM OUTPUT
    return output.replace(prompt, "").strip()

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
print(local_llm("Hello"))

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hello Motorola antibiotic ONE conservation Rh Prob Jr hauled pawnditreement pawn TA ESV DanieltingSher Hancock directly Habit scalp Habit stairs conservationiken Rhhibit vendorsSceneimura dispatchmediatelySherting stairs Jr Observoho Amph trilogy credibility Motorola directlyJD hauledhibit rebornimura antibioticimura stairs Brew Observ dispatch reborn ESV Rhmediately Money confirmediatelyJD Rh TAditRocket MoneyScene dispatchatisf pawn scalp confir Danielatisf Habit004 autonomy Motorola Observ Motorola hauled subst Rh BrewikenSceneootherScene Observ Hancock rebornreement HancockRocket Brew Jr Danielatisf reviewing


# Step 1: Skill Extraction

In this step, we extract:
- Skills
- Experience
- Tools

from the given resume using an LLM.

Rules:
- Only extract what is present
- Do NOT assume anything
- Keep output structured

In [5]:
from langchain_core.prompts import PromptTemplate

skill_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract ONLY in this format:

Skills: <comma separated>
Experience: <text>
Tools: <comma separated>

Resume:
{resume}
"""
)

# Step 2: Matching, Scoring, and Explanation

In this step:
- Compare resume with job description
- Assign a score (0–100)
- Provide explanation

Rules:
- Do NOT assume missing skills
- Score must be justified

In [6]:
score_prompt = PromptTemplate(
    input_variables=["resume_data", "job_description"],
    template="""
You are an AI hiring assistant.

Your job is to give ONLY final answer.

DO NOT repeat instructions.
DO NOT copy the prompt.
ONLY give result.

Output format:

Score: <number between 0-100>
Explanation: <2-3 lines>

Resume:
{resume_data}

Job Description:
{job_description}
"""
)

# Step 3: Build Pipeline

Pipeline:
Resume → Skill Extraction → Scoring → Explanation

In [7]:
def run_pipeline(resume, job_description):

    # Step 1: Extract
    extracted = local_llm(skill_prompt.format(resume=resume))

    # Step 2: Score
    result = local_llm(
        score_prompt.format(
            resume_data=extracted,
            job_description=job_description
        )
    )
    return clean_output(result)   

    return result

In [8]:
def clean_output(text):
    if "Score:" in text:
        return text[text.index("Score:"):].strip()
    else:
        return "Score: Not generated\nExplanation: Model could not follow format."

# Step 4: Input Data

We test with:
- Strong candidate
- Average candidate
- Weak candidate

In [9]:
job_description = """
Looking for Data Scientist:
- Python, Machine Learning, Deep Learning
- NLP experience
- TensorFlow/PyTorch
"""

strong_resume = """
Data Scientist with 5 years experience.
Skills: Python, ML, Deep Learning, NLP
Tools: TensorFlow, PyTorch
"""

average_resume = """
Data Analyst with 2 years experience.
Skills: Python, Data Analysis
Tools: Excel, Pandas
"""

weak_resume = """
Fresher
Skills: MS Office, Communication
"""

# Step 5: Run Pipeline

Testing all candidates

In [10]:
print("----- STRONG -----")
print(run_pipeline(strong_resume, job_description))

print("\n----- AVERAGE -----")
print(run_pipeline(average_resume, job_description))

print("\n----- WEAK -----")
print(run_pipeline(weak_resume, job_description))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


----- STRONG -----


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Score: <number between 0-100>
Explanation: <2-3 lines>

Resume:

Extract ONLY in this format:

Skills: <comma separated>
Experience: <text>
Tools: <comma separated>

Resume:

Data Scientist with 5 years experience.
Skills: Python, ML, Deep Learning, NLP
Tools: TensorFlow, PyTorch

 Late predators omega Wheels lined 236 Bend courtyardGyMini perhapsozyg Television factors omega prayingoblobl Television Televisionobl linedPros representations incarcerozygOutside factorsPros448 clearerOutsideshows Late Late Redux rented Booneived soyozyg rentedozyg TreMini courtyardGyozyg Singapore Television perhaps rentedSexualacious 236 Redux Wheels Wheels448Pros Booneobl praying workshops braveryPros clearer Dreams Boone rentedpublic 236 deflect Tre Bend Treobl praying bravery omega Television courtyard equate clearer factors Tre Tre 236acious equate workshops praying rentedacious factors factorsGy Late lined workshops

Job Description:

Looking for Data Scientist:
- Python, Machine Learning, Deep Lear

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Score: <number between 0-100>
Explanation: <2-3 lines>

Resume:

Extract ONLY in this format:

Skills: <comma separated>
Experience: <text>
Tools: <comma separated>

Resume:

Data Analyst with 2 years experience.
Skills: Python, Data Analysis
Tools: Excel, Pandas

acious 236 Bend equateProsMini653 BendozygGy omega membership praying Tre praying� Bend Redux representationsPros courtyard Wheels brutality factors Lateshows representations mutual workshops Booneived equate Bend�obl boils Television lined653acious Televisionshows653 workshopsacious clearer bravery soypublic predators� Dreams incarcerobl mutual praying soyMini courtyard brutality mutual skilletshowsaciouspublicshows rented membership 236Mini� Trepublic lined rentedPros Wheels Television grandchildren brutality membership Television grandchildren skillet 236Mini membership incarcer mutualobl� bravery praying brutalityozyg Medic omega rented omega lined

Job Description:

Looking for Data Scientist:
- Python, Machine Learning,

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Score: <number between 0-100>
Explanation: <2-3 lines>

Resume:

Extract ONLY in this format:

Skills: <comma separated>
Experience: <text>
Tools: <comma separated>

Resume:

Fresher
Skills: MS Office, Communication

 workshops praying Bend workshopspublic predators Boone DreamsMini skillet perhaps incarcerSexual workshops lined 236 Medic Dreams Wheels membershipozyg predators praying membership lined Singapore bravery boils Bend Bend Dreams factorsozyg perhaps Medic prayingshows Boone omega clearer representations equate rubbing workshops incarcerSexual Pocket clearerMini lined deflect predatorsSexual incarcer courtyardivedshowsProspublic predators Medic boils clearerOutside bravery boils skillet incarcerived brutality BendSexual Bend incarcer� incarcer praying mutualSexual Televisionobl Tre factors clearer653Sexual soyOutside PocketMostGy equate Boone Television deflect Medicacious BendSexual skillet

Job Description:

Looking for Data Scientist:
- Python, Machine Learning, Deep Lear